In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

train_path = '/content/drive/MyDrive/26학년도 1학기/SYNAPSE/난독화된 한글 리뷰 복원 AI 프로젝트/train.csv'
test_path = '/content/drive/MyDrive/26학년도 1학기/SYNAPSE/난독화된 한글 리뷰 복원 AI 프로젝트/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train 데이터 총 개수: {len(train_df)}개")
print(f"Test 데이터 총 개수: {len(test_df)}개")

Train 데이터 총 개수: 11263개
Test 데이터 총 개수: 1689개


In [ ]:
# 데이터프레임에 존재하는 모든 열(컬럼) 이름을 확인하기
print(train_df.columns)

Index(['ID', 'input', 'output'], dtype='object')


In [ ]:
# 실제 컬럼명인 'input'과 'output'에 맞추어 문장 길이 계산
train_df['input_length'] = train_df['input'].apply(lambda x: len(str(x)))
train_df['output_length'] = train_df['output'].apply(lambda x: len(str(x)))

print("=== 1. 평균 문장 길이 분석 ===")
print(f"난독화 리뷰(input) 평균 길이: {train_df['input_length'].mean():.2f}자")
print(f"원문 리뷰(output) 평균 길이: {train_df['output_length'].mean():.2f}자")
print(f"최대 길이(난독화): {train_df['input_length'].max()}자 / 최소 길이(난독화): {train_df['input_length'].min()}자\n")

print("=== 2. 자주 등장하는 난독화 유형 파악용 데이터 10개 출력 ===")
for index, row in train_df.head(10).iterrows():
    print(f"[{index}]")
    print(f"원문  : {row['output']}")
    print(f"난독화: {row['input']}")
    print("-" * 50)

=== 1. 평균 문장 길이 분석 ===
난독화 리뷰(input) 평균 길이: 93.09자
원문 리뷰(output) 평균 길이: 93.12자
최대 길이(난독화): 1381자 / 최소 길이(난독화): 1자

=== 2. 자주 등장하는 난독화 유형 파악용 데이터 10개 출력 ===
[0]
원문  : 별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자니 댓글로는 너무 길고... 아무튼 두 번 다시 가길 싫은 곳. 캠핑을 20여 년 다녀본 곳 중 제일 기분 나빴던 곳.
난독화: 별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈닐 탯끎룐눈 녀뮤 퀼교... 야뭍툰 둠 변 닺씨 깍낄 싫훈 굣. 깸삥읊 20여 년 댜녁뵨 곧 중 쩨윌 귑푼 낙팠떤 곶.
--------------------------------------------------
[1]
원문  : 잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ
난독화: 잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ
--------------------------------------------------
[2]
원문  : 절대 가면 안 되는 곳 메모
난독화: 절테 간면 않 된는 굣 멥몫
--------------------------------------------------
[3]
원문  : 아... 가격 좋고 뷰도 뻥 뚫려서 시원하지만 담배 냄새 미쳐버림. 싸게 하루만 묵겠다! 하는 사람한테만 추천. 담배 냄새가 모든 장점을 가져가는 곳. 노래방에서 각종 담배와 유흥에 쩔었을 때 나는 냄새가 계속 방에 있음 ㅆ... 싸니까 할 말 없음.
난독화: 야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵겠댜! 한눈 쌀람한뗌많 쭈쳔. 탐패 냄쌕갊 묘둔 쟝졈울 까저갼눈 콧. 놂랙팡엣셔 칵좋 닮패왕 윳흥예 천렸욹 택 냐눈 넴쌘갸 꼐쏙 방웨 잊슴 ㅆ... 샨닉깎 할 맑 엽숨.
---------------------------------------------

In [ ]:
# 1. HuggingFace datasets 라이브러리 설치 (코랩 환경 필수)
!pip install datasets

from datasets import Dataset

# 2. Pandas DataFrame에서 필요한 열('input', 'output')만 추출하여 HuggingFace Dataset으로 변환
# (test_df에는 정답인 'output' 열이 없으므로 'input'만 변환)
train_dataset_full = Dataset.from_pandas(train_df[['input', 'output']])
test_dataset = Dataset.from_pandas(test_df[['input']])

# 3. 전체 학습 데이터를 학습용(Train) 80%, 검증용(Validation) 20%로 분할
# seed=42를 주어 팀원 누가 실행해도 항상 똑같이 분할되도록 고정합니다.
split_dataset = train_dataset_full.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset['train']
val_data = split_dataset['test'] # datasets 라이브러리에서는 분할된 20%를 'test'라는 키워드로 저장함 (이것이 검증용 데이터)

print("=== 모델 학습용 데이터셋 변환 및 분할 완료 ===")
print(f"학습용(Train) 데이터 개수: {len(train_data)}개")
print(f"검증용(Validation) 데이터 개수: {len(val_data)}개")
print(f"최종 테스트용(Test) 데이터 개수: {len(test_dataset)}개\n")

print("=== 팀원 전달용 변환 데이터 샘플 확인 ===")
print(train_data[0])

=== 모델 학습용 데이터셋 변환 및 분할 완료 ===
학습용(Train) 데이터 개수: 9010개
검증용(Validation) 데이터 개수: 2253개
최종 테스트용(Test) 데이터 개수: 1689개

=== 팀원 전달용 변환 데이터 샘플 확인 ===
{'input': '청쇼 쌓테 낫뿌쥐 얀았음. 끊뎃 팡움위 념뮤 앉 됨 ㅠ', 'output': '청소 상태 나쁘지 않았음. 근데 방음이 너무 안 됨 ㅠ'}


In [ ]:
import random
from datasets import Dataset
import pandas as pd

# 1. 데이터 증강 함수 (규칙 기반 노이즈 생성)
def add_noise(text):
    if not isinstance(text, str):
        return str(text)

    choice = random.randint(1, 3)
    if choice == 1:
        # 패턴 1: 무작위 특수문자 삽입 ("좋아요" -> "좋@아#요")
        chars = list(text)
        for _ in range(random.randint(1, 3)):
            if len(chars) > 0:
                idx = random.randint(0, len(chars)-1)
                chars.insert(idx, random.choice(['@', '#', '*', '~']))
        return "".join(chars)

    elif choice == 2:
        # 패턴 2: 기존 띄어쓰기 무시 및 무작위 띄어쓰기 ("좋아요" -> "좋 아 요")
        chars = list(text.replace(" ", ""))
        num_spaces = random.randint(1, max(1, len(chars)//3))
        for _ in range(num_spaces):
            if len(chars) > 1:
                idx = random.randint(1, len(chars)-1)
                chars.insert(idx, " ")
        return "".join(chars)

    else:
        # 패턴 3: 반복 문자 삽입 ("좋아요" -> "좋아요ㅋㅋ")
        return text + random.choice(['ㅋㅋ', 'ㅎㅎ', 'ㅠㅠ', '!!'])

# 2. 기존 학습 데이터 중 3000개를 무작위로 뽑아 증강 데이터 생성
aug_sample = train_df.sample(n=3000, random_state=42).copy()
aug_sample['input'] = aug_sample['output'].apply(add_noise)

# 3. 원본 데이터와 증강된 데이터 병합하여 데이터 부풀리기
final_train_df = pd.concat([train_df, aug_sample], ignore_index=True)

# 4. 늘어난 데이터를 HuggingFace Dataset으로 최종 변환 및 분할 (팀원 C 전달용)
final_dataset = Dataset.from_pandas(final_train_df[['input', 'output']])
final_split = final_dataset.train_test_split(test_size=0.2, seed=42)

final_train_data = final_split['train']
final_val_data = final_split['test']

print("=== 데이터 증강 및 최종 모델 입력용 데이터셋 구성 완료 ===")
print(f"증강 전 원본 데이터 개수: {len(train_df)}개")
print(f"증강 후 전체 데이터 개수: {len(final_train_df)}개")
print(f"최종 학습용(Train) 데이터: {len(final_train_data)}개")
print(f"최종 검증용(Validation) 데이터: {len(final_val_data)}개\n")

print("=== 새롭게 증강된 난독화 데이터 샘플 ===")
for index, row in final_train_df.tail(3).iterrows():
    print(f"원문  : {row['output']}")
    print(f"난독화: {row['input']}")
    print("-" * 50)

=== 데이터 증강 및 최종 모델 입력용 데이터셋 구성 완료 ===
증강 전 원본 데이터 개수: 11263개
증강 후 전체 데이터 개수: 14263개
최종 학습용(Train) 데이터: 11410개
최종 검증용(Validation) 데이터: 2853개

=== 새롭게 증강된 난독화 데이터 샘플 ===
원문  : 풋살장 바닥 관리 상태가 별로인데, 가격도 비쌈. 주차는 무료인 건 좋음.
난독화: 풋 살장바 닥관리상태가별로인데,가 격도비쌈.주차는무료인 건좋음.
--------------------------------------------------
원문  : 화장실 배수가 안 됩니다. 수압이 약합니다. 천장에 곰팡이가 있습니다. 방음이 안 됩니다. 청소 상태가 불량합니다. 10만 원 주고 들어갔는데 진짜 최악입니다. 다음엔 차박을 하겠음, 차라리.
난독화: 화장실배 수가안됩니다 .수압이 약합니다. 천장 에곰팡이 가있습니다.방음이  안됩 니다 .청소상태 가불량합니 다.10만원  주고들어갔 는데진짜최 악 입니다.다음엔차   박 을하겠음, 차라리.
--------------------------------------------------
원문  : 아이들과 함께 합니다~ 좋아요~ 물이 진짜 좋아요~ 시설도 깨끗해요~
난독화: 아이들과함께합  니다~좋아 요~   물이진짜 좋아요~시설도깨끗해 요~
--------------------------------------------------
